In [ ]:
# Load dataset from Google Drive (alternative to GitHub clone)
import shutil
from pathlib import Path

drive_zip = '/content/drive/MyDrive/pcb_seg_dataset.zip'
if Path(drive_zip).exists():
    print('Loading dataset from Google Drive...')
    print(f'Zip size: {Path(drive_zip).stat().st_size / 1024 / 1024:.1f} MB')
    
    # Copy zip to local storage for faster extraction
    shutil.copy(drive_zip, '/content/pcb_seg_dataset.zip')
    
    # Extract dataset
    !unzip -q /content/pcb_seg_dataset.zip -d /content/pcb-wafer-inspection-pipeline/01_training/data/processed/
    
    # Verify extraction
    dataset_path = Path('/content/pcb-wafer-inspection-pipeline/01_training/data/processed/pcb_seg')
    if dataset_path.exists():
        train_count = len(list((dataset_path / 'images' / 'train').glob('*.jpg')))
        val_count = len(list((dataset_path / 'images' / 'val').glob('*.jpg')))
        print(f'Dataset loaded successfully: {train_count} train, {val_count} val images')
        
        # Clean up zip file
        Path('/content/pcb_seg_dataset.zip').unlink()
        print('Cleanup: temporary zip file removed')
    else:
        print('ERROR: Dataset extraction failed')
        
else:
    print('Dataset zip not found in Google Drive')
    print('Expected location:', drive_zip)
    print('Either:')
    print('1. Upload pcb_seg_dataset.zip to Google Drive root')
    print('2. Or run the conversion script (next cells will handle raw data)')

## Alternative: Load dataset from Google Drive

**Use this cell instead of re-running conversion if you uploaded the dataset zip to Google Drive.**

Upload `pcb_seg_dataset.zip` to your Google Drive root, then run this cell.

# YOLOv8-seg PCB Defect Detection Training

This notebook trains a YOLOv8n-seg model on PCB defect detection dataset.

**Dataset:** PCB Defect Dataset (6 classes)
- missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper
- 17,366 train objects + 4,298 val objects (bbox-as-polygon format)
- Input: Pascal VOC XML -> Output: YOLO-seg polygon format

**Model:** YOLOv8n-seg.pt (nano, fastest)
**Training:** 100 epochs, batch=16, imgsz=640
**Hardware:** Google Colab T4 GPU

**Expected Results:**
- mAP50 (mask): >0.70 for PCB defects
- mAP50-95 (mask): >0.45
- Training time: ~2-3 hours on T4

**Outputs:**
- best.pt (PyTorch weights)
- best.onnx (ONNX export, opset=21 for OnnxRuntime 1.20.1 compatibility)
- Training logs and validation plots

## Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
print('Google Drive mounted:')
print('  MyDrive contents:', os.listdir('/content/drive/MyDrive')[:10])

## Clone repository from GitHub

In [ ]:
# Clone the project repository
!git clone https://github.com/TaeyangYeon/pcb-wafer-inspection-pipeline.git
%cd pcb-wafer-inspection-pipeline

# Verify project structure
!ls -la 01_training/data/processed/pcb_seg/
print('\nDataset structure verified')

## Install dependencies

In [ ]:
# Install ultralytics (YOLOv8)
!pip install ultralytics

# Verify installation
from ultralytics import YOLO
print('Ultralytics installed successfully')
print('Available models:', ['yolov8n-seg.pt', 'yolov8s-seg.pt', 'yolov8m-seg.pt'])

## Verify GPU availability

In [ ]:
# Check GPU hardware
!nvidia-smi

# Verify PyTorch CUDA
import torch
print('\n=== PyTorch GPU Status ===')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')
else:
    print('WARNING: GPU not available, training will be slow on CPU')

## Override dataset.yaml for Colab path

In [ ]:
# Fix dataset.yaml path for Colab environment
import yaml
from pathlib import Path

yaml_path = Path('01_training/data/processed/pcb_seg/dataset.yaml')
print('Original dataset.yaml:')
with open(yaml_path) as f:
    config = yaml.safe_load(f)
    print('  path:', config['path'])

# Update path for Colab
config['path'] = '/content/pcb-wafer-inspection-pipeline/01_training/data/processed/pcb_seg'
with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print('\nUpdated dataset.yaml:')
print('  path:', config['path'])
print('  nc:', config['nc'])
print('  names:', config['names'])

# Verify dataset files exist
dataset_path = Path(config['path'])
train_count = len(list((dataset_path / 'images' / 'train').glob('*.jpg')))
val_count = len(list((dataset_path / 'images' / 'val').glob('*.jpg')))
print(f'\nDataset verified: {train_count} train, {val_count} val images')

## Train YOLOv8n-seg model

In [ ]:
# Initialize YOLOv8n-seg model
from ultralytics import YOLO
import time

print('Initializing YOLOv8n-seg model...')
model = YOLO('yolov8n-seg.pt')  # Downloads pretrained weights automatically

# Start training
print('Starting training...')
start_time = time.time()

results = model.train(
    data='01_training/data/processed/pcb_seg/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,  # Use GPU
    project='/content/drive/MyDrive/pcb_seg_training',
    name='yolov8n_seg_pcb',
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    patience=20,  # Early stopping patience
    cache=True,  # Cache images for faster training
    workers=2,  # Colab CPU cores
)

training_time = time.time() - start_time
print(f'\nTraining completed in {training_time/3600:.1f} hours')
print('Results saved to:', '/content/drive/MyDrive/pcb_seg_training/yolov8n_seg_pcb')

## Resume Training (use if session disconnected)

Run this cell if your Colab session disconnected during training.
It will automatically find the latest checkpoint and resume training.

In [ ]:
# Resume training from latest checkpoint
from ultralytics import YOLO
import glob
from pathlib import Path

# Find latest checkpoint
checkpoint_dir = '/content/drive/MyDrive/pcb_seg_training/yolov8n_seg_pcb/weights'
if Path(checkpoint_dir).exists():
    checkpoints = sorted(glob.glob(f'{checkpoint_dir}/epoch*.pt'))
    if checkpoints:
        last_ckpt = checkpoints[-1]
        print('Found checkpoints:')
        for ckpt in checkpoints[-3:]:  # Show last 3
            print(f'  {Path(ckpt).name}')
        print(f'\nResuming from: {Path(last_ckpt).name}')
        
        # Resume training
        model = YOLO(last_ckpt)
        results = model.train(resume=True)
        print('Training resumed successfully')
    else:
        print('No epoch checkpoints found')
        print('Available weights:', list(Path(checkpoint_dir).glob('*.pt')))
        print('Run the main training cell (Cell 7) instead')
else:
    print('Training directory not found:', checkpoint_dir)
    print('Run the main training cell (Cell 7) first')

## Evaluate trained model

In [ ]:
# Load best model and evaluate
from ultralytics import YOLO
from pathlib import Path

best_model_path = '/content/drive/MyDrive/pcb_seg_training/yolov8n_seg_pcb/weights/best.pt'

if Path(best_model_path).exists():
    print('Loading best model for evaluation...')
    model = YOLO(best_model_path)
    
    # Run validation
    metrics = model.val(
        data='01_training/data/processed/pcb_seg/dataset.yaml',
        imgsz=640,
        conf=0.001,  # Low conf for mAP calculation
        iou=0.6,
        save_json=True,  # Save results for analysis
    )
    
    print('\n=== Evaluation Results ===')
    print('Detection (Bounding Box):')
    print(f'  mAP50:     {metrics.box.map50:.4f}')
    print(f'  mAP50-95:  {metrics.box.map:.4f}')
    
    print('\nSegmentation (Mask):')
    print(f'  mAP50:     {metrics.seg.map50:.4f}')
    print(f'  mAP50-95:  {metrics.seg.map:.4f}')
    
    print('\nPer-class mAP50 (mask):')
    class_names = ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
    for i, (name, map50) in enumerate(zip(class_names, metrics.seg.map50_class)):
        print(f'  {name:<15}: {map50:.4f}')
        
    # Model size
    model_size = Path(best_model_path).stat().st_size / 1024 / 1024
    print(f'\nModel size: {model_size:.1f} MB')
    
else:
    print('Best model not found:', best_model_path)
    print('Make sure training completed successfully')

## Export to ONNX (opset=21 for compatibility)

In [ ]:
# Export best model to ONNX format
from ultralytics import YOLO
from pathlib import Path

best_model_path = '/content/drive/MyDrive/pcb_seg_training/yolov8n_seg_pcb/weights/best.pt'

if Path(best_model_path).exists():
    print('Loading best model for ONNX export...')
    model = YOLO(best_model_path)
    
    # Export to ONNX with specific opset for compatibility
    print('Exporting to ONNX format...')
    export_path = model.export(
        format='onnx',
        opset=21,  # IMPORTANT: Use opset=21 for OnnxRuntime 1.20.1 compatibility
        simplify=True,
        imgsz=640,
        dynamic=False,  # Static shape for better performance
    )
    
    print('ONNX export completed!')
    print('Export path:', export_path)
    
    # Verify ONNX file
    onnx_size = Path(export_path).stat().st_size / 1024 / 1024
    print(f'ONNX file size: {onnx_size:.1f} MB')
    
    print('\nIMPORTANT NOTES:')
    print('- ONNX opset=21 used (not default 22) for OnnxRuntime 1.20.1 compatibility')
    print('- Model input shape: [1, 3, 640, 640] (NCHW format)')
    print('- Outputs: boxes + masks for 6 PCB defect classes')
    
else:
    print('Best model not found:', best_model_path)
    print('Complete training first')

## Copy outputs to organized Google Drive folder

In [ ]:
# Copy important files to organized folder in Google Drive
import shutil
from pathlib import Path
import datetime

# Source and destination paths
src_dir = Path('/content/drive/MyDrive/pcb_seg_training/yolov8n_seg_pcb')
dst_dir = Path('/content/drive/MyDrive/pcb_seg_outputs')
dst_dir.mkdir(exist_ok=True)

if src_dir.exists():
    # Copy model weights
    weights_src = src_dir / 'weights'
    if (weights_src / 'best.pt').exists():
        shutil.copy(weights_src / 'best.pt', dst_dir / 'best.pt')
        print('Copied: best.pt')
    
    if (weights_src / 'best.onnx').exists():
        shutil.copy(weights_src / 'best.onnx', dst_dir / 'best.onnx')
        print('Copied: best.onnx')
    
    if (weights_src / 'last.pt').exists():
        shutil.copy(weights_src / 'last.pt', dst_dir / 'last.pt')
        print('Copied: last.pt')
    
    # Copy training results
    if (src_dir / 'results.png').exists():
        shutil.copy(src_dir / 'results.png', dst_dir / 'training_results.png')
        print('Copied: training_results.png')
    
    if (src_dir / 'confusion_matrix.png').exists():
        shutil.copy(src_dir / 'confusion_matrix.png', dst_dir / 'confusion_matrix.png')
        print('Copied: confusion_matrix.png')
    
    # Create training summary
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(dst_dir / 'training_summary.txt', 'w') as f:
        f.write(f'YOLOv8n-seg PCB Defect Detection\n')
        f.write(f'Training completed: {timestamp}\n')
        f.write(f'Dataset: PCB 6 classes\n')
        f.write(f'Model: yolov8n-seg.pt\n')
        f.write(f'Epochs: 100\n')
        f.write(f'Batch size: 16\n')
        f.write(f'Image size: 640\n')
        f.write(f'ONNX opset: 21\n')
    print('Created: training_summary.txt')
    
    print('\nFiles saved to Google Drive:')
    for f in dst_dir.iterdir():
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f'  {f.name:<25} {size_mb:>6.1f} MB')
    
    print(f'\nGoogle Drive folder: {dst_dir}')
    print('You can download these files to your local machine')
    
else:
    print('Training directory not found:', src_dir)
    print('Make sure training completed successfully')

## Download Instructions

### To download trained models to your local Mac:

1. **From Google Drive web interface:**
   - Navigate to `My Drive > pcb_seg_outputs`
   - Select `best.pt` and `best.onnx`
   - Right-click > Download

2. **From Colab (alternative):**
   ```python
   from google.colab import files
   files.download('/content/drive/MyDrive/pcb_seg_outputs/best.pt')
   files.download('/content/drive/MyDrive/pcb_seg_outputs/best.onnx')
   ```

3. **Local project integration:**
   - Copy downloaded files to: `01_training/outputs/models/`
   - Update model paths in your inference scripts
   - For .NET C# integration: use `best.onnx` with OnnxRuntime 1.20.1

### Model files:
- **best.pt**: PyTorch weights (for Python inference, fine-tuning)
- **best.onnx**: ONNX model (for C# .NET, cross-platform inference)
- **training_results.png**: Training/validation curves
- **confusion_matrix.png**: Per-class performance analysis

### Expected performance:
- **mAP50 (mask)**: >0.70 for PCB defects
- **mAP50-95 (mask)**: >0.45
- **Inference speed**: ~50-100 FPS on GPU, ~5-15 FPS on CPU
- **Model size**: ~6-12 MB (YOLOv8n is lightweight)